# Определение ориентации текста

Для каждого текстового изображения оценивается вероятность поворота на 180°.
Используется PP-LCNet x1.0 со специализированными исходными весами,
дообучение на HierText и синтетике, объединение двух ориентаций и калибровка.

**Подтверждённый результат на лидерборде: 0.96611242; Brier: 0.03388758.**
Этот результат относится к сохранённым весам
`outputs/finetune_20260920_175555_827888/best.pdparams` и их калибровке.
Более поздний запуск `src.training.train` рассматривается отдельно.

Ноутбук содержит анализ и вызывает готовые модули проекта. По умолчанию
он читает существующие файлы; обучение, загрузка данных и инференс
включаются явно в настройках. Выводы ячеек появятся после запуска на сервере.


## 1. Окружение и настройки

Положите `solution.ipynb` в корень проекта и выберите Python из `.venv`.
Окружение устанавливается командой `uv sync --locked --extra gpu`.
В VS Code достаточно выбрать `.venv/bin/python` как ядро notebook.
Команды из ячеек запускаются тем же интерпретатором, что и ядро.

`SELECTED_MODEL = "confirmed"` выбирает веса отправленного решения;
`"trained"` — веса из `TRAINING_DIR`. Для обучения с нуля укажите новый
`TRAINING_DIR`; существующий запуск с `config.json` продолжается через `--resume`.

В `EXECUTE` можно добавить нужные этапы:
`"prepare"`, `"train"`, `"development"`, `"calibration"`, `"holdout"`, `"predict"`.
Например, `{"predict"}` создаст CSV для выбранных весов с готовой калибровкой.
`set()` достаточно для просмотра уже полученных результатов.


In [ ]:
import hashlib
import json
import shlex
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file() and (path / "src").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Откройте notebook внутри каталога проекта")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.calibrate import apply_temperature
from src.evaluation.metrics import binary_metrics, summarize_metrics
from src.inference.submission import read_sample, validate_submission

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": False})
pd.set_option("display.max_columns", 20)
print(f"Проект: {ROOT}\nИнтерпретатор: {sys.executable}")


In [ ]:
DEVICE = "gpu:0"
BATCH_SIZE = 256
READ_WORKERS = 8
SEED = 42

EXECUTE = set()
SELECTED_MODEL = "confirmed"
TRAINING_DIR = ROOT / "outputs/training"
MANIFEST = ROOT / "data/external/manifest.csv"
TEST_DIR = ROOT / "data/test"

WEIGHTS = {
    "pretrained": ROOT / "weights/pretrained.pdparams",
    "confirmed": ROOT / "outputs/finetune_20260920_175555_827888/best.pdparams",
    "trained": TRAINING_DIR / "best.pdparams",
}
if SELECTED_MODEL not in ("confirmed", "trained"):
    raise ValueError("SELECTED_MODEL: confirmed или trained")
if not EXECUTE <= {"prepare", "train", "development", "calibration", "holdout", "predict"}:
    raise ValueError("В EXECUTE указан неизвестный этап")

SELECTED_WEIGHTS = WEIGHTS[SELECTED_MODEL]
STEM = "finetuned" if SELECTED_MODEL == "confirmed" else TRAINING_DIR.name
EVALUATIONS = {
    split: ROOT / "outputs/evaluation" / f"{STEM}_{split}"
    for split in ("development", "calibration", "holdout")
}
CALIBRATION = ROOT / "outputs/calibration" / f"{STEM}_rotation.json"
OUTPUT_CSV = ROOT / "outputs" / f"submission_{STEM}_notebook.csv"
SCORED_CSV = ROOT / "outputs/submission_calibrated.csv"

CONFIRMED_WEIGHTS_SHA = "ccf2a0cb194cc03b0801aab63fe46cd832bfa6641169ae3b330baabfc58062ba"
CONFIRMED_CSV_SHA = "1b625c7ac0ecd657ac5fff09728c226921e05914f3d7dedb5fc97127aa4692e1"
CONFIRMED_MANIFEST_SHA = "ea3006f74a57d03a5a0003c4b45b83046c1b9a2ef2e8bb42988f628bf58f6e39"


In [ ]:
def read_json(path):
    """Прочитать настройки или отчёт запуска."""
    return json.loads(Path(path).read_text(encoding="utf-8"))


def sha256(path):
    """Посчитать хеш фактического содержимого файла."""
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def run_module(module, *arguments):
    """Запустить модуль в отдельном процессе с выводом журнала в ячейку."""
    command = [sys.executable, "-u", "-m", module, *map(str, arguments)]
    print(shlex.join(command), flush=True)

    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise

    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


packages = ["numpy", "pandas", "opencv-python-headless", "paddlepaddle-gpu", "paddlepaddle"]
environment = []
for package in packages:
    try:
        environment.append({"package": package, "installed": version(package)})
    except PackageNotFoundError:
        environment.append({"package": package, "installed": "не установлен"})
display(pd.DataFrame(environment))


## 2. Задача и протокол оценки

Целевая переменная: `0` — нормальная ориентация, `1` — поворот на 180°.
Для каждого подготовленного кропа создаются обе ориентации, поэтому
число примеров в метриках вдвое больше числа исходных PNG.

$$\operatorname{Brier}=\frac{1}{N}\sum_{i=1}^{N}(p_i-y_i)^2,
\qquad \operatorname{score}=1-\operatorname{Brier}.$$

Accuracy показывает число верных решений при пороге 0.5, но не оценивает
качество вероятностей. Основной критерий выбора — Brier.

| Часть | Назначение |
| --- | --- |
| train | Обновление параметров модели |
| development | Выбор весов и режима инференса |
| calibration | Подбор температуры для выбранных весов |
| holdout | Оценка зафиксированной конфигурации |

Сцены и найденные группы дубликатов не разделяются между частями.
Метрики реальных и синтетических примеров считаются отдельно.
Тестовые изображения соревнования используются только для инференса.


In [ ]:
# Контроль определения метрики: постоянная вероятность 0.5.
binary_metrics([0, 1], [0.5, 0.5])


## 3. Подготовленный корпус

[HierText](https://github.com/google-research-datasets/hiertext) предоставляет
аннотации слов, строк и абзацев. Здесь используется его архив validation;
собственные train/development/calibration/holdout создаются внутри выбранного
внешнего корпуса и не являются официальными частями бенчмарка HierText.

Подготовка фильтрует аннотации, выпрямляет кропы, создаёт синтетические строки
и объединяет найденные дубликаты. Шрифты сохраняются в `data/fonts/`,
кэш исходников — в `data/cache/`, данные для модели — в `data/external/`.
RusTitW поддерживается отдельным источником, но в этом эксперименте не использовался.

Повторная подготовка ниже включается только через `EXECUTE`. Флаг очистки
`--rebuild` не используется. `applied_rotation` — исправление исходного кропа
при подготовке; это поле нельзя использовать как целевую метку обучения.


In [ ]:
if "prepare" in EXECUTE:
    run_module("src.data.prepare", "--sources", "hiertext", "synthetic",
               "--output", MANIFEST.parent)

if not MANIFEST.is_file():
    raise FileNotFoundError(f"Подготовьте данные: {MANIFEST}")

frame = pd.read_csv(MANIFEST, keep_default_na=False)
MANIFEST_SHA = sha256(MANIFEST)
required = {"crop_id", "path", "scene_id", "group_id", "split", "source",
            "dataset", "script", "text", "width", "height"}
if not required <= set(frame.columns) or frame.empty:
    raise ValueError("Не подходит формат manifest.csv; нужен текущий src.data.prepare")

if frame.crop_id.duplicated().any() or frame.path.duplicated().any():
    raise ValueError("В manifest повторяются ID или пути")
for column in ("scene_id", "group_id"):
    if frame.groupby(column).split.nunique().max() != 1:
        raise ValueError(f"Обнаружено пересечение частей по {column}")

print(f"Кропов: {len(frame):,}; SHA-256 manifest: {MANIFEST_SHA}")
print("Совпадает с корпусом подтверждённого запуска:", MANIFEST_SHA == CONFIRMED_MANIFEST_SHA)
display(frame[["crop_id", "source", "script", "width", "height", "split"]].head())


In [ ]:
SPLIT_ORDER = ["train", "development", "calibration", "holdout"]
summary = frame.groupby("split").agg(
    crops=("crop_id", "size"), scenes=("scene_id", "nunique"), groups=("group_id", "nunique")
).reindex(SPLIT_ORDER)
summary["orientation_examples"] = 2 * summary.crops
display(summary)

script_counts = pd.crosstab([frame.split, frame.source], frame.script)
display(script_counts)

counts = pd.crosstab(frame.split, frame.source).reindex(SPLIT_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
counts.plot.bar(stacked=True, ax=axes[0], color=["#3366a3", "#df9444"], rot=20)
axes[0].set(title="Исходные кропы по частям", xlabel="", ylabel="Кропов")

for source, values in frame.loc[frame.split.eq("train")].groupby("source"):
    bins = np.geomspace(1, max(2, frame.height.max() + 1), 35)
    axes[1].hist(values.height, bins=bins, alpha=0.55, label=source)
axes[1].set(xscale="log", title="Высота train-кропов", xlabel="Высота, пиксели", ylabel="Кропов")
axes[1].legend()
plt.show()


В зафиксированном корпусе 70 271 исходный кроп: 66 271 HierText и 4 000 синтетических.
Чисто кириллических реальных кропов всего 50, и все они оказались в train.
В остальных частях встречаются лишь единичные смешанные тексты.
Поэтому результат на синтетической кириллице не заменяет оценку на реальном русском тексте.

Ниже показаны примеры **только из train**. Для каждого изображения справа
создаётся класс `1`; при обучении и оценке тот же поворот выполняется до resize.


In [ ]:
train_frame = frame.loc[frame.split.eq("train")]
examples = pd.concat([
    group.sample(n=1, random_state=SEED)
    for _, group in train_frame.groupby(["source", "script"], sort=True)
]).head(6)

fig, axes = plt.subplots(len(examples), 2, figsize=(12, 1.9 * len(examples)),
                         squeeze=False, layout="constrained")
for row_axes, row in zip(axes, examples.itertuples(), strict=True):
    with Image.open(MANIFEST.parent / row.path) as image:
        pixels = np.asarray(image.convert("RGB"))
    for target, view in enumerate((pixels, pixels[::-1, ::-1])):
        row_axes[target].imshow(view)
        row_axes[target].set_title(f"{row.source} / {row.script}; y={target}; {row.text[:45]}")
        row_axes[target].axis("off")
plt.show()


## 4. Модель и объединение ориентаций

В `src/models/lcnet.py` находится адаптация PP-LCNet x1.0 из
[PaddleClas](https://github.com/PaddlePaddle/PaddleClas).
Используются специализированные обучаемые веса `textline_ori`;
их загрузка и проверка совместимости реализованы в `src.models`.
Текущий цикл дообучения написан в `src.training.train`.

Обработка входа: RGB → resize до **160 × 80** → нормализация ImageNet → CHW, FP32.
Размер задаётся как ширина × высота. `preprocessing.py` используется без изменения.

В режиме `single` берётся вероятность класса `1`. Режим `rotation` объединяет
исходное изображение и его поворот:

$$q(x)=\frac{p(x)+1-p(\operatorname{rot180}(x))}{2}.$$

Такая формула согласует прогнозы пары: $q(\operatorname{rot180}(x))=1-q(x)$.
Уже к объединённой вероятности применяется температура:

$$p_{180}(x)=\operatorname{sigmoid}(\operatorname{logit}(q(x))/T).$$

Температура меняет уверенность; при положительной $T$ решения при пороге 0.5
сохраняются. Формулы используются готовыми модулями оценки и инференса.


## 5. Дообучение и выбор эпохи

Текущий рецепт: 6 эпох, batch size 256, LR 0.001, cross-entropy,
Momentum 0.9 с Nesterov, L2 `4e-5`, ограничение нормы градиента до 5.
Разогрев занимает примерно четверть эпохи; затем cosine снижает LR до 0.0001.
Веса выбираются по **real-development Brier / rotation / T = 1**.

Обе ориентации train-кропов хранятся в RAM как uint8; сам кэш занимает 3.49 GiB.
Дополнительные отражения и произвольные повороты не применяются.
Сохраняются завершённые эпохи, оптимизатор и настройки; `--resume` продолжает тот же план.
Незавершённая эпоха повторяется. Побитовая идентичность CUDA не гарантируется.

Подтверждённые веса были получены отдельным более ранним запуском: тогда эпоха 4
выбиралась по cross-entropy на всём development. Новый модуль также изменил
порядок поворота/resize и установку seed по эпохам. Поэтому новый запуск
не следует представлять как точное повторение траектории прежнего обучения.


In [ ]:
if "train" in EXECUTE:
    if (TRAINING_DIR / "config.json").is_file():
        arguments = ["--output", TRAINING_DIR, "--resume"]
    else:
        if not WEIGHTS["pretrained"].is_file():
            from src.models.weights import download_pretrained
            download_pretrained(WEIGHTS["pretrained"])
        arguments = ["--manifest", MANIFEST, "--weights", WEIGHTS["pretrained"],
                     "--output", TRAINING_DIR, "--epochs", 6, "--batch-size", BATCH_SIZE,
                     "--lr", 0.001, "--seed", SEED, "--selection-variant", "rotation"]

    run_module("src.training.train", *arguments,
               "--read-workers", READ_WORKERS, "--device", DEVICE)
else:
    print("Обучение отключено; ниже читаются сохранённые результаты")


In [ ]:
history_path = TRAINING_DIR / "history.json"
history_table = pd.DataFrame()
trained_best = None

if history_path.is_file():
    history = read_json(history_path)
    training_config = read_json(TRAINING_DIR / "config.json")
    trained_best = read_json(TRAINING_DIR / "best.json")
    if training_config["manifest_sha256"] != MANIFEST_SHA:
        raise ValueError("История обучения относится к другому manifest")
    if sha256(WEIGHTS["trained"]) != trained_best["weights_sha256"]:
        raise ValueError("best.json не соответствует весам обучения")

    history_table = pd.DataFrame([
        {key: item[key] for key in ("epoch", "train_loss", "selection_brier", "seconds")}
        for item in history
    ])
    best_row = history_table.loc[history_table.selection_brier.idxmin()]
    if int(best_row.epoch) != trained_best["epoch"]:
        raise ValueError("Выбранная эпоха расходится с минимумом Brier в истории")
    display(history_table.round(8))

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), layout="constrained")
    axes[0].plot(history_table.epoch, history_table.train_loss, "o-", color="#3366a3")
    axes[0].set(title="Обучение", xlabel="Эпоха", ylabel="Train cross-entropy")
    axes[1].plot(history_table.epoch, history_table.selection_brier, "o-", color="#df9444")
    axes[1].axvline(trained_best["epoch"], color="#444444", ls="--", label="Выбранная эпоха")
    axes[1].set(title="Real-development / rotation", xlabel="Эпоха", ylabel="Brier ↓")
    axes[1].legend()
    plt.show()
else:
    print(f"История обучения отсутствует: {history_path}")


В присланном завершённом запуске новый модуль выбрал эпоху **5**:
Brier на real-development — **0.02217974**. У checkpoint подтверждённого
сабмишена в том же режиме — **0.02077580**. Поэтому для итогового инференса
сохранены подтверждённые веса. Оценка нового checkpoint на лидерборде пока отсутствует.
Таблица и графики выше читаются из текущего каталога обучения; при новом запуске
их значения могут измениться.


## 6. Сравнение на development

Готовые отчёты проверяются по хешам весов и manifest: результаты разных
корпусов или checkpoint нельзя объединять в одну таблицу сравнения.
Без калибровки модуль сохраняет оба режима — `single` и `rotation`.
Основное сравнение проводится на реальных кропах; синтетика показана отдельно.

Если нужной оценки ещё нет, добавьте `"development"` в `EXECUTE` и выполните
ячейки с начала. Существующие отчёты повторно не рассчитываются.


In [ ]:
def read_evaluation(directory, weights, split, temperature=1.0):
    """Прочитать оценку и проверить, каким весам и данным она соответствует."""
    if not directory.exists():
        print(f"Нет сохранённой оценки: {directory.relative_to(ROOT)}")
        return None

    report = read_json(directory / "report.json")
    expected = {"weights_sha256": sha256(weights),
                "manifest_sha256": MANIFEST_SHA, "split": split}
    for key, value in expected.items():
        if report.get(key) != value:
            raise ValueError(f"{directory}: не совпадает {key}")
    if not np.isclose(report["temperature"], temperature, rtol=0, atol=1e-12):
        raise ValueError(f"{directory}: не совпадает температура")

    metrics = pd.read_csv(directory / "metrics.csv")
    return {"directory": directory, "report": report, "metrics": metrics}


def create_evaluation(weights, split, directory, calibration=None):
    """Получить отсутствующую оценку через общий CLI проекта."""
    if directory.exists():
        return
    arguments = ["--weights", weights, "--manifest", MANIFEST,
                 "--split", split, "--output", directory, "--batch-size", BATCH_SIZE,
                 "--read-workers", READ_WORKERS, "--device", DEVICE]
    if calibration is not None:
        arguments += ["--calibration", calibration]
    run_module("src.evaluation.evaluate", *arguments)


if SELECTED_MODEL == "confirmed" and SELECTED_WEIGHTS.is_file():
    if sha256(SELECTED_WEIGHTS) != CONFIRMED_WEIGHTS_SHA:
        raise ValueError("По пути подтверждённого checkpoint лежат другие веса")


In [ ]:
development_results = {}
comparison_parts = []
development_paths = {
    "pretrained": ROOT / "outputs/evaluation/pretrained_development",
    SELECTED_MODEL: EVALUATIONS["development"],
}

for name, directory in development_paths.items():
    if "development" in EXECUTE:
        create_evaluation(WEIGHTS[name], "development", directory)
    result = read_evaluation(directory, WEIGHTS[name], "development")
    if result is not None:
        development_results[name] = result
        comparison_parts.append(result["metrics"].assign(model=name, origin="evaluation"))

if trained_best is not None and "trained" not in development_results:
    best_record = next(item for item in history if item["epoch"] == trained_best["epoch"])
    comparison_parts.append(pd.DataFrame(best_record["development"]).assign(
        model="trained", origin="history.json"
    ))

if comparison_parts:
    comparison = pd.concat(comparison_parts, ignore_index=True)
    columns = ["model", "source", "variant", "crops", "n", "brier", "score", "accuracy", "origin"]
    display(comparison[columns].sort_values(["source", "brier"]).round(8))

    real_comparison = comparison.loc[comparison.source.eq("real")]
    chart = real_comparison.pivot(index="model", columns="variant", values="brier")
    ax = chart.plot.bar(figsize=(9, 3.8), color=["#3366a3", "#df9444"], rot=0)
    ax.set(title="Development: только реальные кропы, T = 1", xlabel="", ylabel="Brier ↓")
    plt.tight_layout()
    plt.show()


## 7. Разбор ошибок на development

Здесь используются прогнозы выбранных весов в режиме `rotation`, до калибровки.
Показаны срезы по письменности и размерам, затем кропы с наибольшей квадратичной
ошибкой. Каждая исходная картинка появляется в галерее один раз.

В колонке `n` учитываются обе ориентации. Они зависимы, поэтому `n` нельзя
трактовать как число независимых сцен. Выводы по редким группам требуют осторожности.
Holdout для подбора исправлений в этом разделе не используется.


In [ ]:
development_predictions = None
selected_development = development_results.get(SELECTED_MODEL)

if selected_development is not None:
    predictions = pd.read_csv(
        selected_development["directory"] / "predictions.csv.gz", keep_default_na=False
    )
    development_predictions = predictions.loc[
        predictions.source.eq("real") & predictions.variant.eq("rotation")
    ].copy()
    development_predictions["height_slice"] = np.where(
        development_predictions.height <= 32, "height <= 32", "height > 32"
    )
    development_predictions["aspect_slice"] = np.where(
        development_predictions.width / development_predictions.height > 10,
        "width / height > 10", "width / height <= 10"
    )

    slices = []
    for column in ("script", "height_slice", "aspect_slice"):
        for name, values in development_predictions.groupby(column):
            slices.append({"slice": column, "value": name,
                           "crops": values.crop_id.nunique(),
                           **binary_metrics(values.target, values.p_180)})
    display(pd.DataFrame(slices).round(8))
else:
    print("Для разбора ошибок нужна сохранённая оценка выбранной модели на development")


In [ ]:
if development_predictions is not None:
    errors = development_predictions.assign(
        squared_error=lambda values: (values.p_180 - values.target) ** 2
    ).sort_values("squared_error", ascending=False).drop_duplicates("crop_id").head(6)

    fig, axes = plt.subplots(2, 3, figsize=(13, 5), layout="constrained")
    for ax in axes.flat:
        ax.axis("off")
    for ax, row in zip(axes.flat, errors.itertuples()):
        with Image.open(MANIFEST.parent / row.path) as image:
            pixels = np.asarray(image.convert("RGB"))
        if row.target == 1:
            pixels = pixels[::-1, ::-1]
        ax.imshow(pixels)
        ax.set_title(f"y={row.target}, p₁₈₀={row.p_180:.3f}, error={row.squared_error:.3f}\n"
                     f"{row.script}; {row.text[:35]}", fontsize=10)
    plt.show()


## 8. Температурная калибровка

После выбора модели и режима температура подбирается по Brier только на
**реальных кропах calibration**. Для подтверждённого checkpoint получено
`T = 1.1390818601472776`: Brier на этой части изменился с 0.02258039 до 0.02252003.
Это результат на данных подбора, а не независимая оценка улучшения.

Сохранённая температура используется повторно. Для другого checkpoint нужна
своя калибровка. Включение `"calibration"` рассчитывает только отсутствующие артефакты.


In [ ]:
if "calibration" in EXECUTE:
    create_evaluation(SELECTED_WEIGHTS, "calibration", EVALUATIONS["calibration"])
    if not CALIBRATION.is_file():
        run_module("src.evaluation.calibrate", "--evaluation", EVALUATIONS["calibration"],
                   "--variant", "rotation", "--output", CALIBRATION)

calibration = None
calibration_evaluation = None
if CALIBRATION.is_file():
    calibration = read_json(CALIBRATION)
    expected = {"weights_sha256": sha256(SELECTED_WEIGHTS),
                "manifest_sha256": MANIFEST_SHA, "fit_source": "real",
                "fit_split": "calibration", "variant": "rotation"}
    for key, value in expected.items():
        if calibration.get(key) != value:
            raise ValueError(f"Калибровка не соответствует выбранным данным и весам: {key}")
    calibration_evaluation = read_evaluation(
        EVALUATIONS["calibration"], SELECTED_WEIGHTS, "calibration"
    )
    print(f"Температура: {calibration['temperature']:.10f}")
    display(pd.DataFrame({"before": calibration["before"],
                          "after": calibration["after"]}).T.round(8))
else:
    print(f"Нет калибровки выбранных весов: {CALIBRATION.relative_to(ROOT)}")


Диаграмма ниже группирует прогнозы по интервалам вероятности.
По горизонтали — средняя предсказанная вероятность класса `1`, по вертикали —
его фактическая доля. Диагональ соответствует совпадению этих величин.
Размер точек отражает число примеров в интервале. Это описательная проверка
на calibration; при малом числе примеров отклонения могут быть нестабильны.


In [ ]:
if calibration_evaluation is not None:
    cal_predictions = pd.read_csv(
        calibration_evaluation["directory"] / "predictions.csv.gz", keep_default_na=False
    )
    real_cal = cal_predictions.loc[
        cal_predictions.source.eq("real") & cal_predictions.variant.eq("rotation")
    ]
    before = real_cal.p_180.to_numpy()
    after = apply_temperature(before, calibration["temperature"])

    fig, ax = plt.subplots(figsize=(5.5, 5), layout="constrained")
    ax.plot([0, 1], [0, 1], "--", color="#777777", label="Диагональ")
    for name, values, color in (("До", before, "#3366a3"), ("После", after, "#df9444")):
        actual = binary_metrics(real_cal.target, values)
        recorded = calibration["before" if name == "До" else "after"]
        if not np.isclose(actual["brier"], recorded["brier"], rtol=0, atol=1e-10):
            raise ValueError("Прогнозы calibration не соответствуют метрикам в JSON")

        bins = np.minimum((values * 10).astype(int), 9)
        reliability = pd.DataFrame({"bin": bins, "probability": values,
                                    "target": real_cal.target.to_numpy()}).groupby("bin").agg(
            probability=("probability", "mean"), rate=("target", "mean"), n=("target", "size")
        )
        ax.plot(reliability.probability, reliability.rate, color=color, alpha=0.6)
        ax.scatter(reliability.probability, reliability.rate, color=color, label=name,
                   s=20 + 100 * reliability.n / reliability.n.max())
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Среднее p₁₈₀", ylabel="Доля класса 1",
           title="Real-calibration / rotation")
    ax.legend()
    plt.show()


## 9. Оценка зафиксированной конфигурации

На holdout применяются выбранные веса, `rotation` и уже сохранённая температура.
Для подтверждённой конфигурации получено:

| Источник | Кропов | Примеров | Brier | 1 − Brier |
| --- | ---: | ---: | ---: | ---: |
| Real | 6 750 | 13 500 | 0.02628072 | 0.97371928 |
| Synthetic | 375 | 750 | 0.00339131 | 0.99660869 |

Эти результаты уже просмотрены. При дальнейшей настройке нельзя представлять
этот holdout как ранее не исследованную выборку или выбирать на нём температуру.
Ячейка ниже читает готовую оценку; вычисление включается через `"holdout"`.


In [ ]:
holdout_evaluation = None
if calibration is not None:
    if "holdout" in EXECUTE:
        create_evaluation(SELECTED_WEIGHTS, "holdout", EVALUATIONS["holdout"], CALIBRATION)
    holdout_evaluation = read_evaluation(
        EVALUATIONS["holdout"], SELECTED_WEIGHTS, "holdout", calibration["temperature"]
    )
    if holdout_evaluation is not None:
        display(holdout_evaluation["metrics"].round(8))
elif "holdout" in EXECUTE:
    raise FileNotFoundError("Сначала нужна калибровка выбранных весов")


## 10. Итоговый CSV

Инференс использует `src.inference.predict`. Включите `"predict"` в `EXECUTE`,
чтобы создать файл `OUTPUT_CSV`: путь отличается от уже отправленного CSV.
Модуль проверяет совпадение калибровки с весами и сохраняет JSON с хешами и временем.
Существующие файлы не перезаписываются автоматически.

Тестовые изображения ожидаются в `data/test/test/images/`, образец —
в `data/test/sample_submission.csv`. Порядок строк берётся из образца.
Проверяются 20 000 ID и конечные вероятности от 0 до 1.
Без тестовых меток локально нельзя посчитать качество сабмишена.


In [ ]:
if "predict" in EXECUTE:
    if calibration is None:
        raise FileNotFoundError("Для итогового режима нужна калибровка выбранных весов")
    if not OUTPUT_CSV.exists():
        run_module("src.inference.predict", "--weights", SELECTED_WEIGHTS,
                   "--calibration", CALIBRATION, "--test-dir", TEST_DIR,
                   "--batch-size", BATCH_SIZE, "--read-workers", READ_WORKERS,
                   "--device", DEVICE, "--output", OUTPUT_CSV)

submission_path = OUTPUT_CSV if OUTPUT_CSV.is_file() else None
if submission_path is None and SELECTED_MODEL == "confirmed" and SCORED_CSV.is_file():
    submission_path = SCORED_CSV

if submission_path is None:
    print("Сохранённого CSV нет; для создания включите predict в EXECUTE")
else:
    sample = read_sample(TEST_DIR / "sample_submission.csv")
    submission = pd.read_csv(submission_path, dtype={"image_id": str})
    validate_submission(submission, sample)
    submission_report = read_json(submission_path.with_suffix(".json"))
    csv_hash = sha256(submission_path)
    if submission_report["output_sha256"] != csv_hash:
        raise ValueError("CSV не соответствует хешу в отчёте")
    if submission_report["weights_sha256"] != sha256(SELECTED_WEIGHTS):
        raise ValueError("CSV получен другими весами")
    if submission_report["sample_sha256"] != sha256(TEST_DIR / "sample_submission.csv"):
        raise ValueError("CSV создан с другим образцом ответа")
    if calibration is None or submission_report["variant"] != "rotation" or not np.isclose(
        submission_report["temperature"], calibration["temperature"], rtol=0, atol=1e-12
    ):
        raise ValueError("CSV не соответствует выбранной калибровке")

    display(submission.head())
    print(f"CSV: {submission_path.relative_to(ROOT)}; строк: {len(submission)}")
    print(f"Диапазон p_180: {submission.p_180.min():.10f} … {submission.p_180.max():.10f}")
    print(f"PNG → CSV: {submission_report['png_to_csv_seconds']:.3f} с")
    print(f"SHA-256: {csv_hash}")
    if csv_hash == CONFIRMED_CSV_SHA:
        print("Это подтверждённый CSV: LB score = 0.96611242; Brier = 0.03388758")
    else:
        print("Этот CSV отличается от отправленного; результат на LB ему не приписывается")


## 11. Результат и ограничения

Зафиксированный сабмишен использует дообученные веса, `rotation` и температуру
**1.1390818601472776**. На лидерборде получено **0.96611242** против **0.95800658**
у предыдущего отправленного CSV: разница **+0.00810584**. Это совместный эффект
изменений инференса и калибровки; отдельный эффект температуры на LB не измерялся.

В записанном запуске на RTX 6000 Ada обработка 20 000 PNG и сохранение CSV
заняли **17.357 с** при batch size 256 и восьми потоках чтения.
Загрузка модели и прогрев в замер не входят.

Основные ограничения:

- Реальных кириллических данных мало; синтетика не подтверждает качество на реальном русском тексте.
- Сцены объединяются по обнаруженным дубликатам. Хеши не гарантируют обнаружение всех кадрирований и сложных преобразований.
- Внешний корпус и тест соревнования отличаются; их метрики не обязаны совпадать.
- Holdout уже просмотрен. Дальнейшее сравнение моделей проводится на development, температура подбирается на calibration.
- Новый модуль обучения проверен завершённым запуском, но его checkpoint имеет другой Brier и пока не получил собственной оценки на LB.

Следующие содержательные эксперименты: отдельная проверка реальных русскоязычных
кропов, анализ трудных групп development и подключение RusTitW после проверки
его разметки. Эти эксперименты не входят в подтверждённый результат.

Для переноса результата нужно сохранить вместе веса, JSON калибровки, CSV,
соседний JSON отчёта и данные оценки. `outputs/` и `weights/` исключены из Git;
одного исходного кода недостаточно для восстановления конкретного checkpoint.
